# YOLOv6S Drone Detection Training (Enhanced Dataset)
**Model:** YOLOv6s (small - accurate)
**Datasets Combined:**
- Roboflow: Zhejiang University drones (4,231 images)
- Kaggle: drone-dataset-uav
- Kaggle: yolo-drone-detection-dataset
**Target:** OAK-1W (416x416, FP16 blob)
**Training:** 100 epochs

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone YOLOv6 and install dependencies
!git clone https://github.com/meituan/YOLOv6.git /kaggle/working/YOLOv6
%cd /kaggle/working/YOLOv6
!pip install -r requirements.txt -q
!pip install roboflow -q

In [ ]:
# 3. Download Kaggle datasets (already authenticated in Kaggle)
import os

os.makedirs("/kaggle/working/datasets", exist_ok=True)

# Download Kaggle datasets
!kaggle datasets download -d dasmehdixtr/drone-dataset-uav -p /kaggle/working/datasets/drone-uav --unzip
!kaggle datasets download -d muki2003/yolo-drone-detection-dataset -p /kaggle/working/datasets/yolo-drone --unzip

# Download Roboflow dataset
from roboflow import Roboflow
rf = Roboflow(api_key="KbQOzd2sc4AJ40krFJZa")
project = rf.workspace("zhejiang-university-china-dliq1").project("drones-detection-with-yolov8")
dataset = project.version(1).download("yolov5", location="/kaggle/working/datasets/roboflow-drones")

print("\nDatasets downloaded!")

In [ ]:
# 4. Merge datasets into YOLO format
import os
import shutil
from pathlib import Path

BASE = Path("/kaggle/working/datasets")
MERGED = BASE / "merged"
TRAIN_IMG = MERGED / "images" / "train"
VAL_IMG = MERGED / "images" / "val"
TRAIN_LBL = MERGED / "labels" / "train"
VAL_LBL = MERGED / "labels" / "val"

for p in [TRAIN_IMG, VAL_IMG, TRAIN_LBL, VAL_LBL]:
    p.mkdir(parents=True, exist_ok=True)

def copy_dataset(src_img_dir, src_lbl_dir, dst_img, dst_lbl, split_ratio=0.9):
    images = list(Path(src_img_dir).glob("*.jpg")) + list(Path(src_img_dir).glob("*.png"))
    n_train = int(len(images) * split_ratio)
    
    for i, img_path in enumerate(images):
        dst_i = dst_img if i < n_train else VAL_IMG
        dst_l = dst_lbl if i < n_train else VAL_LBL
        
        shutil.copy(img_path, dst_i / img_path.name)
        lbl_path = Path(src_lbl_dir) / (img_path.stem + ".txt")
        if lbl_path.exists():
            shutil.copy(lbl_path, dst_l / lbl_path.name)

# 1. Roboflow dataset (already split)
roboflow_train = BASE / "roboflow-drones" / "train"
roboflow_val = BASE / "roboflow-drones" / "valid"

for img in (roboflow_train / "images").glob("*"):
    shutil.copy(img, TRAIN_IMG / img.name)
for lbl in (roboflow_train / "labels").glob("*"):
    shutil.copy(lbl, TRAIN_LBL / lbl.name)
for img in (roboflow_val / "images").glob("*"):
    shutil.copy(img, VAL_IMG / img.name)
for lbl in (roboflow_val / "labels").glob("*"):
    shutil.copy(lbl, VAL_LBL / lbl.name)

print(f"Roboflow: {len(list(TRAIN_IMG.glob('*')))} train, {len(list(VAL_IMG.glob('*')))} val")

# 2. drone-dataset-uav
uav_dirs = list((BASE / "drone-uav").glob("**/images"))
if uav_dirs:
    for img_dir in uav_dirs[:3]:
        lbl_dir = img_dir.parent / "labels"
        if lbl_dir.exists():
            copy_dataset(img_dir, lbl_dir, TRAIN_IMG, TRAIN_LBL)

print(f"After UAV: {len(list(TRAIN_IMG.glob('*')))} train, {len(list(VAL_IMG.glob('*')))} val")

# 3. yolo-drone-detection-dataset
yolo_dirs = list((BASE / "yolo-drone").glob("**/images"))
if yolo_dirs:
    for img_dir in yolo_dirs[:3]:
        lbl_dir = img_dir.parent / "labels"
        if lbl_dir.exists():
            copy_dataset(img_dir, lbl_dir, TRAIN_IMG, TRAIN_LBL)

print(f"Final: {len(list(TRAIN_IMG.glob('*')))} train, {len(list(VAL_IMG.glob('*')))} val images")

In [ ]:
# 5. Create dataset YAML and verify
yaml_content = f'''train: /kaggle/working/datasets/merged/images/train
val: /kaggle/working/datasets/merged/images/val
nc: 1
names: ['drone']
'''

with open("/kaggle/working/YOLOv6/data/drones_merged.yaml", "w") as f:
    f.write(yaml_content)

train_count = len(list(Path("/kaggle/working/datasets/merged/images/train").glob("*")))
val_count = len(list(Path("/kaggle/working/datasets/merged/images/val").glob("*")))
print(f"Training images: {train_count}")
print(f"Validation images: {val_count}")

In [ ]:
# 6. Patch torch.load for PyTorch 2.6+ and train
import os

with open("/kaggle/working/sitecustomize.py", "w") as f:
    f.write('''
import torch as _torch
_orig_load = _torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
_torch.load = _patched_load
''')

print("Patch written. Starting training...")
os.environ["PYTHONPATH"] = "/kaggle/working"

!cd /kaggle/working/YOLOv6 && PYTHONPATH=/kaggle/working python tools/train.py \
    --batch 32 \
    --conf configs/yolov6s.py \
    --data data/drones_merged.yaml \
    --img-size 416 \
    --epochs 100 \
    --name drone_yolov6s \
    --device 0 2>&1

In [ ]:
# 7. Export to ONNX
!pip install onnxscript -q
!cd /kaggle/working/YOLOv6 && PYTHONPATH=/kaggle/working python deploy/ONNX/export_onnx.py \
    --weights runs/train/drone_yolov6s/weights/best_ckpt.pt \
    --img-size 416 416 \
    --batch-size 1 \
    --simplify

In [ ]:
# 8. Convert ONNX to .blob for OAK-1W
!pip install blobconverter -q

import blobconverter

blob_path = blobconverter.from_onnx(
    model="/kaggle/working/YOLOv6/runs/train/drone_yolov6s/weights/best_ckpt.onnx",
    data_type="FP16",
    shaves=6,
    version="2022.1",
    output_dir="/kaggle/working",
    optimizer_params=[
        "--input_shape=[1,3,416,416]",
        "--data_type=FP16",
    ],
)

print(f"Blob saved to: {blob_path}")

In [ ]:
# 9. Copy outputs for download
import shutil

shutil.copy("/kaggle/working/YOLOv6/runs/train/drone_yolov6s/weights/best_ckpt.pt", "/kaggle/working/best_yolov6s.pt")
print("Output files:")
!ls -lh /kaggle/working/best_yolov6s.pt /kaggle/working/*.blob